### Python Script: Deep Clustering (DEC-style)

In [0]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

# 1. Define a Simple Autoencoder for Dimensionality Reduction
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return latent, reconstructed

# 2. Generate Synthetic High-Dimensional Data
data, _ = make_blobs(n_samples=1000, n_features=50, centers=5, random_state=42)
data_tensor = torch.FloatTensor(data)

# 3. Train the Autoencoder to learn the latent space
model = Autoencoder(input_dim=50, latent_dim=10)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

for epoch in range(100):
    latent, reconstructed = model(data_tensor)
    loss = criterion(reconstructed, data_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# 4. Perform Clustering in the Latent Space (Deep Clustering)
with torch.no_grad():
    latent_features, _ = model(data_tensor)
    kmeans = KMeans(n_clusters=5, n_init=10)
    clusters = kmeans.fit_predict(latent_features.numpy())

print(f"Assigned cluster labels for first 5 points: {clusters[:5]}")


#### 1. Advanced Image Clustering Script


#### When to Choose TORC over K-Means

- Irregular Shapes: If your clusters are "C-shaped" or elongated, TORC will group them correctly, whereas K-Means would likely split them into spheres.
- Noisy Datasets: TORC includes a "Self-Correction" stage that identifies and labels outliers as 0 instead of forcing them into a cluster.
- Unknown Structure: Use TORC when you have no idea how many categories exist in your data (e.g., fraud detection or biological pattern discovery). 

In [0]:
import numpy as np
from scipy.spatial.distance import cdist
from TorqueClustering import TorqueClustering # Ensure the library is in your path

# 1. Prepare your data
# Example: 500 samples with 10 features each
data = np.random.rand(500, 10)

# 2. Compute a Distance Matrix (DM)
# DM is an (n x n) matrix showing distance between every pair of points
DM = cdist(data, data, metric='euclidean')

# 3. Run TORC in Automatic Mode
# Parameters: (DistanceMatrix, K=0 for auto, isnoise=True, isfig=True)
idx, idx_with_noise = TorqueClustering(DM, K=0, isnoise=True, isfig=True)

# 4. View Results
print(f"Number of clusters found automatically: {len(np.unique(idx))}")
print(f"Cluster labels: {idx}")
